### 3.2 Improving SSMs with Selection

Consider an input sequence

$$
x \in \mathbb{R}^{B \times L \times D},
$$

where:

- $B$: batch size.
- $L$: sequence length.
- $D$: feature (embedding) dimension.
- $N$: hidden state dimension of the SSM.

The main idea of Mamba is to make the SSM parameters depend on the input, allowing the model to dynamically decide how information should be processed.

In **S4**, the parameters are fixed throughout the entire sequence:

$$
A,\; B,\; C,\; \Delta = \text{learnable parameters}
$$

where:

- $A$: state transition matrix, controlling how the hidden state evolves.
- $B$: input projection matrix, determining how the input updates the hidden state.
- $C$: output projection matrix, mapping the hidden state to the output.
- $\Delta$: discretization step size.

After discretization,

$$
(\overline{A}, \overline{B}) = \mathrm{discretize}(A, B, \Delta),
$$

the same dynamics are applied at every time step. Therefore, S4 is **time-invariant** and can be efficiently implemented as either a recurrence or a convolution.

In **Mamba (Selective SSM)**, the parameters become functions of the current input:

$$
B = s_B(x), \qquad
C = s_C(x), \qquad
\Delta = s_\Delta(x),
$$

where

$$
s_B(x)=\mathrm{Linear}_N(x), \qquad
s_C(x)=\mathrm{Linear}_N(x),
$$

and

$$
s_\Delta(x)=\mathrm{Broadcast}_D(\mathrm{Linear}_1(x)).
$$

The resulting value of $\Delta$ is passed through a Softplus activation,

$$
\tau_\Delta = \mathrm{softplus},
$$

ensuring that $\Delta > 0$.

Since these parameters are generated for every input token, their shapes change from

$$
B,C:(D,N)
$$

to

$$
B,C:(B,L,N),
$$

and

$$
\Delta:(D)
\rightarrow
(B,L,D).
$$

For example, if a sequence contains $L=100$ tokens, the model generates 100 different sets of parameters:

$$
\{B_1,\ldots,B_{100}\}, \qquad
\{C_1,\ldots,C_{100}\}, \qquad
\{\Delta_1,\ldots,\Delta_{100}\}.
$$

Thus, each token is processed with its own state-space dynamics instead of sharing the same parameters across the entire sequence.
This makes the model **time-varying**, meaning each position in the sequence has its own state-space dynamics. Consequently, the convolution formulation is no longer valid, and the sequence must be processed through a recurrent scan.

<div>
    <img src='../images/SELECTIONMAMBA.png' width="800">
</div>

### Memory-Efficient Scan

Instead of storing all input-dependent SSM parameters in GPU **HBM (High Bandwidth Memory)**, Mamba processes the sequence in small chunks using **SRAM (Shared Memory)**, which is much faster but has limited capacity.

For each chunk, the model:

1. Loads the required SSM parameters from **HBM** to **SRAM**.
2. Performs **discretization**.
3. Executes the **recurrent scan** entirely in **SRAM**.
4. Writes only the final outputs back to **HBM**.
5. Repeats the process for the next chunk while carrying the hidden state.

```text
HBM (large, slow)
      │
      ▼
Load a chunk to SRAM
      │
      ▼
SRAM (small, fast)
 ├── Discretization
 ├── Recurrent Scan
 ├── Update Hidden State
 └── Compute Outputs
      │
      ▼
Write outputs to HBM
      │
      ▼
Repeat for the next chunk
```

Since only one chunk is processed at a time, the entire sequence never needs to fit into SRAM. This significantly reduces memory traffic between HBM and SRAM, improving GPU efficiency.

## Mamba block

<div>
    <img src="../images/MAMBABLOCK.png" width="400">
<div>


### Parameter Count of the Linear Projections in Mamba

Assume the model uses:

$$
D = 512
$$

and an expansion factor

$$
E = 2.
$$

The inner dimension of the Mamba block becomes

$$
ED = 2 \times 512 = 1024.
$$

Visually:

```text
Input
512 dimensions
```

↓

```text
Inner representation
1024 dimensions
```

↓

```text
Output
512 dimensions
```

---

### 1. First Input Projection

This corresponds to the **left green Linear block**.

It projects

$$
512
\rightarrow
1024
$$

Therefore, the weight matrix is

$$
W_1 \in \mathbb{R}^{1024 \times 512}.
$$

Number of parameters:

$$
1024 \times 512 = 524,288.
$$

---

### 2. Second Input Projection

This is the **right green Linear block**.

It receives the same input and performs exactly the same transformation:

$$
512
\rightarrow
1024.
$$

Its weight matrix is

$$
W_2 \in \mathbb{R}^{1024 \times 512}.
$$

Number of parameters:

$$
1024 \times 512 = 524,288.
$$

At this point we have:

```text
Linear 1
524,288 parameters

Linear 2
524,288 parameters
```

Total:

```text
1,048,576 parameters
```

---

### 3. Output Projection

After the Selective SSM and the gating operation, the representation still has

```text
1024 dimensions
```

However, the next Mamba block expects an embedding of size

```text
512 dimensions.
```

Therefore, a final projection is applied:

$$
1024
\rightarrow
512.
$$

The corresponding weight matrix is

$$
W_3 \in \mathbb{R}^{512 \times 1024}.
$$

Number of parameters:

$$
512 \times 1024 = 524,288.
$$

---

### 4. Total Number of Parameters

The three linear projections contribute:

```text
Main input projection

512 → 1024
524,288 parameters

+

Gate input projection

512 → 1024
524,288 parameters

+

Output projection

1024 → 512
524,288 parameters
```

Therefore,

```text
524,288
+524,288
+524,288
-------------
1,572,864 parameters
```

---

### 5. Why Does the Paper Write $3ED^2$?

Each projection contains

$$
ED^2
$$

parameters because

$$
ED^2 = (E \times D)\times D.
$$

Substituting the values

$$
E = 2,
$$

$$
D = 512,
$$

gives

$$
ED^2
=
2 \times 512^2.
$$

Since

$$
512^2 = 262,144,
$$

we obtain

$$
ED^2
=
2 \times 262,144
=
524,288.
$$

This is exactly the number of parameters in **one** linear projection.

Since there are **three** linear projections, the total is

$$
3ED^2
=
3 \times 524,288
=
1,572,864.
$$

---

## Visual Summary

```text
                 Input (512)
                      │
        ┌─────────────┴─────────────┐
        │                           │
        ▼                           ▼
 Linear 1                     Linear 2
512 → 1024                  512 → 1024
524,288                     524,288
        │                           │
     Conv+SSM                    SiLU
        │                           │
        └───────────×───────────────┘
                    │
                    ▼
            Linear Output
           1024 → 512
            524,288
                    │
                    ▼
               Output (512)
```

### Key Insight

The paper expresses the parameter count as

$$
3ED^2
$$

because it generalizes to **any model size**.

For example, if a larger model uses

- $D = 768$
- $E = 4$

the same formula still applies without repeating the entire calculation.

Thus, the expression

$$
3ED^2
$$

simply represents the total number of parameters contributed by the **three linear projections** of a Mamba block.